In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv(r'C:\Users\met00546\formation\Projet data analysis\cards_data.csv')

In [ ]:
print(df.head())

In [ ]:
print(df.info())

In [ ]:
print(df.describe())

In [ ]:
df['acct_open_date'] = pd.to_datetime(df['acct_open_date'])

In [ ]:
df['expires'] = pd.to_datetime(df['expires'])

In [ ]:
# Conserver uniquement mois + année sous forme Period (sans jour)
df[['acct_open_date', 'expires']] = (
    df[['acct_open_date', 'expires']]
    .apply(lambda col: col.dt.to_period('M'))
)


In [ ]:
df.isnull().sum()

In [ ]:
print(df['card_type'].value_counts())

In [ ]:
print(df['card_brand'].value_counts())

In [ ]:
print(df['client_id'].nunique())

In [ ]:
df.shape

In [ ]:
df.groupby('client_id').size().describe()


In [ ]:
df['id'].is_unique


In [ ]:
df.groupby('client_id').size().describe()


In [ ]:
df['card_number'] = df['card_number'].astype(str)
df['card_number'].str.len().value_counts()


In [ ]:
pd.crosstab(df['card_brand'], df['card_number'].str.len())


In [ ]:
pd.crosstab(df['card_brand'], df['cvv'].astype(str).str.len())


In [ ]:
df['cvv_len'] = df['cvv'].astype(str).str.len()

df.groupby('card_brand')['cvv_len'].value_counts(normalize=True)

In [ ]:
df['cvv_len'] = df['cvv'].astype(str).str.len()

df.groupby('card_type')['cvv_len'].value_counts(normalize=True)


In [ ]:
df['cvv_len'] = df['cvv'].astype(str).str.len()

In [ ]:
cvv_distribution = df['cvv_len'].value_counts().sort_index()
print(cvv_distribution)

In [ ]:
df = df[df['cvv_len'] == 3].copy()

In [ ]:
df.drop(columns=['cvv_len'], inplace=True)


In [ ]:
(df['expires'] <= df['acct_open_date']).sum()


In [ ]:
df.loc[df['expires'] <= df['acct_open_date'], 
       ['client_id', 'acct_open_date', 'expires']].head()


In [ ]:
df = df[df['expires'] > df['acct_open_date']].copy()


In [ ]:
df['year_pin_last_changed'].describe()


In [ ]:
df['num_cards_issued'].value_counts().sort_index()


In [ ]:

df.groupby('card_brand')['credit_limit'].describe()

In [ ]:
df['credit_limit'] = (
    df['credit_limit']
      .str.replace('$', '', regex=False)
      .astype(float)
)

In [ ]:
df.groupby('card_type')['credit_limit'].describe()

In [ ]:
df['card_on_dark_web'].value_counts(normalize=True)


In [ ]:
df.drop(columns=['card_on_dark_web'], inplace=True)


In [ ]:
pd.crosstab(df['card_brand'], df['has_chip'])


In [ ]:
df.duplicated(
    subset=['client_id', 'card_number', 'expires']
).sum()


In [ ]:
print(df.info())

In [ ]:
print(df.head())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Préparer les données ---
current_period = pd.Period('2024-01', freq='M')

# Ancienneté des cartes en mois
df['card_age_months'] = (current_period.year - df['acct_open_date'].dt.year) * 12 + \
                        (current_period.month - df['acct_open_date'].dt.month)

# Mois restants avant expiration
df['months_to_expiry'] = (df['expires'].dt.year - current_period.year) * 12 + \
                         (df['expires'].dt.month - current_period.month)

# --- KPI principaux ---
total_cards = df.shape[0]
total_clients = df['client_id'].nunique()
avg_cards_per_client = total_cards / total_clients
avg_credit_limit = df['credit_limit'].mean()
chip_rate = (df['has_chip'] == 'YES').mean() * 100

# --- Créer un dataframe KPI pour Excel ---
kpi_df = pd.DataFrame({
    'KPI': ['Total cartes', 'Total clients', 'Cartes par client', 'Credit limit moyen', 'Taux cartes avec puce'],
    'Valeur': [total_cards, total_clients, avg_cards_per_client, avg_credit_limit, chip_rate]
})

# Exporter les KPI dans Excel
kpi_df.to_excel("kpi_dashboard.xlsx", index=False)

# --- Créer la figure dashboard ---
fig, axes = plt.subplots(3, 2, figsize=(18, 15))
fig.suptitle('Dashboard Carte Bancaire', fontsize=20)

# Afficher les KPI directement sur la figure
kpi_text = f'''
Total cartes       : {total_cards}
Total clients      : {total_clients}
Cartes/client      : {avg_cards_per_client:.2f}
Credit limit moyen : ${avg_credit_limit:.2f}
Taux cartes avec puce : {chip_rate:.1f}%
'''
fig.text(0.02, 0.98, kpi_text, fontsize=12, va='top', ha='left', bbox=dict(facecolor='lightgrey', alpha=0.3))

# --- Graphiques ---
df['card_brand'].value_counts().plot(kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Nombre de cartes par Brand')
axes[0,0].set_xlabel('Card Brand')
axes[0,0].set_ylabel('Nombre de cartes')

df['card_type'].value_counts().plot(kind='bar', ax=axes[0,1], color='lightgreen')
axes[0,1].set_title('Nombre de cartes par Type')
axes[0,1].set_xlabel('Card Type')
axes[0,1].set_ylabel('Nombre de cartes')

df.groupby('card_brand')['credit_limit'].mean().plot(kind='bar', ax=axes[1,0], color='salmon')
axes[1,0].set_title('Credit Limit moyen par Brand')
axes[1,0].set_xlabel('Card Brand')
axes[1,0].set_ylabel('Credit Limit ($)')

df.groupby('card_brand')['card_age_months'].mean().plot(kind='bar', ax=axes[1,1], color='orange')
axes[1,1].set_title("Ancienneté moyenne des cartes (mois)")
axes[1,1].set_xlabel('Card Brand')
axes[1,1].set_ylabel('Mois')

pd.crosstab(df['card_brand'], df['has_chip'], normalize='index').plot(
    kind='bar', stacked=True, ax=axes[2,0], colormap='Pastel1')
axes[2,0].set_title('Taux de cartes avec puce par Brand')
axes[2,0].set_xlabel('Card Brand')
axes[2,0].set_ylabel('Proportion')

df['months_to_expiry'].plot(kind='hist', bins=20, ax=axes[2,1], color='violet')
axes[2,1].set_title('Histogramme des mois avant expiration')
axes[2,1].set_xlabel('Mois restants')
axes[2,1].set_ylabel('Nombre de cartes')

plt.tight_layout(rect=[0.03, 0, 1, 0.95])

# --- Exporter le dashboard en PNG pour présentation ---
plt.savefig("dashboard_carte.png", dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Ton dataframe df déjà nettoyé
current_period = pd.Period('2024-01', freq='M')

# Ancienneté des cartes en mois
df['card_age_months'] = (current_period.year - df['acct_open_date'].dt.year) * 12 + \
                        (current_period.month - df['acct_open_date'].dt.month)

# Mois restants avant expiration
df['months_to_expiry'] = (df['expires'].dt.year - current_period.year) * 12 + \
                         (df['expires'].dt.month - current_period.month)

# KPI
total_cards = df.shape[0]
total_clients = df['client_id'].nunique()
avg_cards_per_client = total_cards / total_clients
avg_credit_limit = df['credit_limit'].mean()
chip_rate = (df['has_chip'] == 'YES').mean() * 100

# 1️⃣ Nombre de cartes par Brand
df_brand_count = df['card_brand'].value_counts().reset_index()
df_brand_count.columns = ['Card Brand', 'Nombre de cartes']
fig_brand = px.bar(df_brand_count, x='Card Brand', y='Nombre de cartes', title='Nombre de cartes par Brand')

# 2️⃣ Nombre de cartes par Type
df_type_count = df['card_type'].value_counts().reset_index()
df_type_count.columns = ['Card Type', 'Nombre de cartes']
fig_type = px.bar(df_type_count, x='Card Type', y='Nombre de cartes', title='Nombre de cartes par Type')

# 3️⃣ Limite de crédit moyenne par Brand
df_credit = df.groupby('card_brand')['credit_limit'].mean().reset_index()
fig_credit = px.bar(df_credit, x='card_brand', y='credit_limit', 
                    title='Limite de crédit moyenne par Brand',
                    labels={'credit_limit':'Limite de crédit ($)', 'card_brand':'Brand'})

# 4️⃣ Ancienneté moyenne par Brand
df_age = df.groupby('card_brand')['card_age_months'].mean().reset_index()
fig_age = px.bar(df_age, x='card_brand', y='card_age_months', 
                 title='Ancienneté moyenne des cartes par Brand',
                 labels={'card_age_months':'Ancienneté (mois)', 'card_brand':'Brand'})

# 5️⃣ Histogramme des mois avant expiration
fig_expiry = px.histogram(df, x='months_to_expiry', nbins=20,
                          title='Mois restants avant expiration',
                          labels={'months_to_expiry':'Mois restants', 'count':'Nombre de cartes'})

# 6️⃣ Proportion cartes avec puce par Brand
chip_pct = pd.crosstab(df['card_brand'], df['has_chip'], normalize='index') * 100
fig_chip = go.Figure()
for col in chip_pct.columns:
    fig_chip.add_trace(go.Bar(
        x=chip_pct.index,
        y=chip_pct[col],
        name=col
    ))
fig_chip.update_layout(barmode='stack', title='Proportion cartes avec puce par Brand', yaxis=dict(title='Pourcentage (%)'))


In [ ]:
# dashboard_carte.py

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, html, dcc

# -------------------------
# 1️⃣ Préparer les données
# -------------------------

# Supposons que df est ton DataFrame déjà nettoyé
# df = pd.read_csv("tes_donnees.csv")  # si besoin

current_period = pd.Period('2024-01', freq='M')

# Ancienneté des cartes en mois
df['card_age_months'] = (current_period.year - df['acct_open_date'].dt.year) * 12 + \
                        (current_period.month - df['acct_open_date'].dt.month)

# Mois restants avant expiration
df['months_to_expiry'] = (df['expires'].dt.year - current_period.year) * 12 + \
                         (df['expires'].dt.month - current_period.month)

# KPI principaux
total_cards = df.shape[0]
total_clients = df['client_id'].nunique()
avg_cards_per_client = total_cards / total_clients
avg_credit_limit = df['credit_limit'].mean()
chip_rate = (df['has_chip'] == 'YES').mean() * 100

# -------------------------
# 2️⃣ Graphiques
# -------------------------

# Nombre de cartes par Brand
df_brand_count = df['card_brand'].value_counts().reset_index()
df_brand_count.columns = ['Card Brand', 'Nombre de cartes']
fig_brand = px.bar(df_brand_count,
                   x='Card Brand',
                   y='Nombre de cartes',
                   title='Nombre de cartes par Brand')

# Nombre de cartes par Type
df_type_count = df['card_type'].value_counts().reset_index()
df_type_count.columns = ['Card Type', 'Nombre de cartes']
fig_type = px.bar(df_type_count,
                  x='Card Type',
                  y='Nombre de cartes',
                  title='Nombre de cartes par Type')

# Limite de crédit moyenne par Brand
df_credit = df.groupby('card_brand')['credit_limit'].mean().reset_index()
fig_credit = px.bar(df_credit,
                    x='card_brand',
                    y='credit_limit',
                    title='Limite de crédit moyenne par Brand',
                    labels={'credit_limit':'Limite de crédit ($)', 'card_brand':'Brand'})

# Ancienneté moyenne par Brand
df_age = df.groupby('card_brand')['card_age_months'].mean().reset_index()
fig_age = px.bar(df_age,
                 x='card_brand',
                 y='card_age_months',
                 title='Ancienneté moyenne des cartes par Brand',
                 labels={'card_age_months':'Ancienneté (mois)', 'card_brand':'Brand'})

# Histogramme des mois avant expiration
fig_expiry = px.histogram(df,
                          x='months_to_expiry',
                          nbins=20,
                          title='Mois restants avant expiration',
                          labels={'months_to_expiry':'Mois restants', 'count':'Nombre de cartes'})

# Proportion cartes avec puce par Brand
chip_pct = pd.crosstab(df['card_brand'], df['has_chip'], normalize='index') * 100
fig_chip = go.Figure()
for col in chip_pct.columns:
    fig_chip.add_trace(go.Bar(
        x=chip_pct.index,
        y=chip_pct[col],
        name=col
    ))
fig_chip.update_layout(barmode='stack',
                       title='Proportion cartes avec puce par Brand',
                       yaxis=dict(title='Pourcentage (%)'))

# -------------------------
# 3️⃣ Création du dashboard Dash
# -------------------------

app = Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Dashboard Carte Bancaire', style={'textAlign':'center'}),

    # KPI en haut
    html.Div([
        html.Div([html.H3('Total cartes'), html.P(f"{total_cards}")],
                 style={'display':'inline-block','width':'20%','textAlign':'center'}),
        html.Div([html.H3('Total clients'), html.P(f"{total_clients}")],
                 style={'display':'inline-block','width':'20%','textAlign':'center'}),
        html.Div([html.H3('Moyenne cartes/client'), html.P(f"{avg_cards_per_client:.2f}")],
                 style={'display':'inline-block','width':'20%','textAlign':'center'}),
        html.Div([html.H3('Credit Limit moyen ($)'), html.P(f"{avg_credit_limit:.2f}")],
                 style={'display':'inline-block','width':'20%','textAlign':'center'}),
        html.Div([html.H3('Taux cartes avec puce (%)'), html.P(f"{chip_rate:.1f}%")],
                 style={'display':'inline-block','width':'20%','textAlign':'center'}),
    ], style={'padding':'10px','borderBottom':'2px solid #ccc'}),

    # Graphiques
    html.Div([
        dcc.Graph(figure=fig_brand),
        dcc.Graph(figure=fig_type),
        dcc.Graph(figure=fig_credit),
        dcc.Graph(figure=fig_age),
        dcc.Graph(figure=fig_expiry),
        dcc.Graph(figure=fig_chip)
    ])
])

# -------------------------
# 4️⃣ Lancer le dashboard
# -------------------------

if __name__ == '__main__':
    app.run(debug=True)  # Dash v3+


In [161]:
figures = [
    ('Nombre de cartes par Brand', fig_brand),
    ('Nombre de cartes par Type', fig_type),
    ('Limite de crédit moyenne par Brand', fig_credit),
    ('Ancienneté moyenne des cartes par Brand', fig_age),
    ('Mois restants avant expiration', fig_expiry),
    ('Proportion cartes avec puce par Brand', fig_chip)
]


In [162]:
from pathlib import Path

html_content = "<html><head><title>Dashboard Carte Bancaire</title></head><body>"
html_content += "<h1 style='text-align:center'>Dashboard Carte Bancaire</h1>"

# Ajouter les KPI
html_content += f"""
<div style='display:flex; justify-content:space-around; padding:10px; border-bottom:2px solid #ccc;'>
    <div><h3>Total cartes</h3><p>{total_cards}</p></div>
    <div><h3>Total clients</h3><p>{total_clients}</p></div>
    <div><h3>Moyenne cartes/client</h3><p>{avg_cards_per_client:.2f}</p></div>
    <div><h3>Credit Limit moyen ($)</h3><p>{avg_credit_limit:.2f}</p></div>
    <div><h3>Taux cartes avec puce (%)</h3><p>{chip_rate:.1f}%</p></div>
</div>
"""

# Ajouter chaque figure
for title, fig in figures:
    html_content += f"<h2 style='text-align:center'>{title}</h2>"
    html_content += fig.to_html(full_html=False, include_plotlyjs='cdn')

html_content += "</body></html>"

# Écrire dans un fichier HTML
output_file = Path("dashboard_carte.html")
output_file.write_text(html_content, encoding='utf-8')

print(f"Dashboard exporté dans {output_file.resolve()}")


Dashboard exporté dans C:\Users\met00546\formation\data analyst\dashboard_carte.html
